# DJIA Low-Level DSP Feature Extraction

## Single Track Analysis: Marrakech (Hermanez)

This notebook demonstrates feature extraction using **low-level DSP functions** directly,
without the high-level `extract_track_features()` wrapper.

**Goal:** Understand how each DSP engine works step-by-step.

---

In [1]:
# Step 1: Load audio with librosa
import librosa
import numpy as np
from pathlib import Path

# Find first audio file
data_dir = Path('data')
audio_files = sorted(data_dir.glob('*.mp3'))[:1]

if not audio_files:
    print('ERROR: No MP3 files found in data/')
else:
    audio_path = audio_files[0]
    print(f'Selected: {audio_path.name}')
    
    # Load audio with librosa
    # sr=22050: standard sample rate for analysis
    # mono=True: convert to mono (1 channel)
    y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
    
    duration = librosa.get_duration(y=y, sr=sr)
    
    print(f'\nAudio Properties:')
    print(f'  Sample Rate: {sr} Hz')
    print(f'  Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)')
    print(f'  Samples: {len(y):,}')
    print(f'  Channels: 1 (mono)')

Selected:  Hermanez - Marrakech.mp3


c:\Users\l_ace\Desktop\projects\djia\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Audio Properties:
  Sample Rate: 22050 Hz
  Duration: 635.22 seconds (10.59 minutes)
  Samples: 14,006,540
  Channels: 1 (mono)


## Phase 2A: Groove Engine (Step 2)

**Goal:** Extract BPM, beat grid, and swing characteristics

**Functions used:**
- `librosa.beat.onset_strength()` - detect beat onsets
- `librosa.beat.tempo()` - estimate BPM
- `analyze_groove()` - complete groove analysis

In [2]:
# Step 2A: Extract Groove (BPM, beats, swing)
from src.dsp.groove_engine import analyze_groove

print('Analyzing Groove Engine...')
print('  - Detecting beat onsets')
print('  - Estimating BPM')
print('  - Computing beat grid')
print('  - Calculating swing score')
print()

# Call low-level groove analysis function
groove_result = analyze_groove(y, sr, hop_length=512)

print('Groove Results:')
print(f'  BPM (decimal): {groove_result.bpm:.2f}')
print(f'  Beats detected: {len(groove_result.beat_times)}')
print(f'  Swing score: {groove_result.swing_score:.3f} (0=stiff, 1=groovy)')
print(f'  Tempo stable: {groove_result.tempo_stability}')
print(f'  Tempo variance: {groove_result.stability_variance:.4f}')
print(f'\nFirst 10 beat times:')
for i, beat_time in enumerate(groove_result.beat_times[:10]):
    print(f'    Beat {i+1}: {beat_time:.2f}s')

Analyzing Groove Engine...
  - Detecting beat onsets
  - Estimating BPM
  - Computing beat grid
  - Calculating swing score

Groove Results:
  BPM (decimal): 129.20
  Beats detected: 1346
  Swing score: 1.000 (0=stiff, 1=groovy)
  Tempo stable: True
  Tempo variance: 0.0009

First 10 beat times:
    Beat 1: 0.53s
    Beat 2: 1.00s
    Beat 3: 1.46s
    Beat 4: 1.95s
    Beat 5: 2.41s
    Beat 6: 2.88s
    Beat 7: 3.37s
    Beat 8: 3.83s
    Beat 9: 4.32s
    Beat 10: 4.78s


## Phase 2B: Mood Engine (Step 3)

**Goal:** Extract harmonic content (key, brightness)

**Functions used:**
- `librosa.feature.chroma_cqt()` - extract chroma features
- `librosa.feature.spectral_centroid()` - compute brightness
- `analyze_mood()` - complete mood analysis

In [3]:
# Step 2B: Extract Mood (key, brightness)
from src.dsp.mood_engine import analyze_mood

print('Analyzing Mood Engine...')
print('  - Extracting chroma features')
print('  - Detecting musical key')
print('  - Computing spectral brightness')
print()

# Call low-level mood analysis function
mood_result = analyze_mood(y, sr)

print('Mood Results:')
print(f'  Musical Key: {mood_result.key}')
print(f'  Camelot Key: {mood_result.camelot_key}')
print(f'  Brightness: {mood_result.brightness:.3f} (0=dark, 1=bright)')
print(f'  Key Confidence: {mood_result.key_confidence:.3f}')

Analyzing Mood Engine...
  - Extracting chroma features
  - Detecting musical key
  - Computing spectral brightness

Mood Results:
  Musical Key: F#/Gb minor
  Camelot Key: 4A
  Brightness: 0.164 (0=dark, 1=bright)
  Key Confidence: 0.078


## Phase 2C: Phrasing Engine (Step 1)

**Goal:** Extract structural segments and hot cue positions

**Functions used:**
- `librosa.stft()` - compute spectrogram
- `compute_novelty_curve()` - measure spectral changes
- `detect_segment_boundaries()` - find peaks
- `create_segments()` - create labeled segments
- `analyze_structure()` - complete phrasing analysis

In [4]:
# Step 2C: Extract Phrasing (segments, structure, cues)
from src.dsp.phrasing_engine import analyze_structure

print('Analyzing Phrasing Engine...')
print('  - Computing STFT')
print('  - Calculating spectral novelty curve')
print('  - Detecting segment boundaries')
print('  - Creating segments and auto-labeling')
print('  - Mapping hot cues')
print()

# Call low-level phrasing analysis function
# Pass BPM from groove engine for proper segment labeling
phrasing_result = analyze_structure(y, sr, bpm=groove_result.bpm, hop_length=512)

print('Phrasing Results:')
print(f'  Segment boundaries detected: {len(phrasing_result.segment_boundaries)}')
print(f'  Total segments created: {len(phrasing_result.segments)}')
print(f'  Hot cues mapped: {len(phrasing_result.cue_points)}')
print(f'  Structure confidence: {phrasing_result.structure_confidence:.2f}')
print(f'\nFirst 10 segments:')
for i, seg in enumerate(phrasing_result.segments[:10], 1):
    duration = seg.end_time - seg.start_time
    bar_start = (seg.start_time * groove_result.bpm) / 60 / 4
    bar_end = (seg.end_time * groove_result.bpm) / 60 / 4
    print(f'  {i:2d}. {seg.label:10s} {seg.start_time:7.2f}s → {seg.end_time:7.2f}s '
          f'({duration:6.2f}s, bars {bar_start:6.2f}-{bar_end:6.2f}, conf {seg.confidence:.0%})')

Analyzing Phrasing Engine...
  - Computing STFT
  - Calculating spectral novelty curve
  - Detecting segment boundaries
  - Creating segments and auto-labeling
  - Mapping hot cues

Phrasing Results:
  Segment boundaries detected: 93
  Total segments created: 94
  Hot cues mapped: 91
  Structure confidence: 0.80

First 10 segments:
   1. intro         0.00s →    0.70s (  0.70s, bars   0.00-  0.38, conf 80%)
   2. breakdown     0.70s →    4.95s (  4.25s, bars   0.38-  2.66, conf 80%)
   3. breakdown     4.95s →   10.15s (  5.20s, bars   2.66-  5.46, conf 80%)
   4. breakdown    10.15s →   14.40s (  4.25s, bars   5.46-  7.75, conf 80%)
   5. breakdown    14.40s →   20.06s (  5.67s, bars   7.75- 10.80, conf 80%)
   6. breakdown    20.06s →   25.26s (  5.20s, bars  10.80- 13.60, conf 80%)
   7. breakdown    25.26s →   29.51s (  4.25s, bars  13.60- 15.89, conf 80%)
   8. breakdown    29.51s →   35.18s (  5.67s, bars  15.89- 18.94, conf 80%)
   9. breakdown    35.18s →   40.38s (  5.20s, bar

## Phase 2D: Curation Engine (Step 4)

**Goal:** Extract danceability, energy, and semantic features

**Functions used:**
- `librosa.feature.zero_crossing_rate()` - percussiveness
- `librosa.feature.tempogram()` - rhythm strength
- `librosa.feature.spectral_centroid()` - brightness
- `analyze_curation()` - complete curation analysis

In [5]:
# Step 2D: Extract Curation (danceability, energy, tags)
from src.dsp.curation_engine import analyze_curation

print('Analyzing Curation Engine...')
print('  - Computing zero-crossing rate (percussiveness)')
print('  - Extracting tempogram (rhythm strength)')
print('  - Analyzing spectral content')
print('  - Generating semantic tags')
print()

# Call low-level curation analysis function
# Note: Curation engine requires data from other engines
curation_result = analyze_curation(
    y, sr,
    bpm=groove_result.bpm,
    swing_score=groove_result.swing_score,
    brightness=mood_result.brightness,
    hop_length=512
)

print('Curation Results:')
print(f'  Danceability: {curation_result.danceability:.3f} (0-1 scale)')
print(f'  Energy type: {curation_result.energy_type}')
print(f'  Complexity score: {curation_result.complexity_score:.3f}')
print(f'  Semantic tags: {', '.join(curation_result.semantic_tags)}')

Analyzing Curation Engine...
  - Computing zero-crossing rate (percussiveness)
  - Extracting tempogram (rhythm strength)
  - Analyzing spectral content
  - Generating semantic tags

Curation Results:
  Danceability: 0.604 (0-1 scale)
  Energy type: dynamic
  Complexity score: 0.505
  Semantic tags: moderate-energy, peak-heavy, groovy, dark, techno, minimalist


## Summary: Complete DSP Feature Extraction

All 4 DSP engines have been applied to the Marrakech track using **low-level functions**.

In [6]:
# Summary of all extracted features
print('\n' + '='*80)
print('COMPLETE FEATURE EXTRACTION FOR: Marrakech (Hermanez)')
print('='*80)

print('\n[GROOVE ENGINE - Rhythm Analysis]')
print(f'  BPM: {groove_result.bpm:.2f} (decimal precision)')
print(f'  Beats: {len(groove_result.beat_times)} detected')
print(f'  Swing: {groove_result.swing_score:.2f} (0=stiff, 1=groovy)')
print(f'  Tempo Stable: {groove_result.tempo_stability}')

print('\n[MOOD ENGINE - Harmonic Analysis]')
print(f'  Musical Key: {mood_result.key}')
print(f'  Camelot Key: {mood_result.camelot_key}')
print(f'  Brightness: {mood_result.brightness:.2f} (0=dark, 1=bright)')
print(f'  Key Confidence: {mood_result.key_confidence:.2f}')

print('\n[PHRASING ENGINE - Structure Analysis]')
print(f'  Total Segments: {len(phrasing_result.segments)}')
print(f'  Segment Types: ', end='')
types = {}
for seg in phrasing_result.segments:
    types[seg.label] = types.get(seg.label, 0) + 1
print(', '.join(f'{t}={n}' for t, n in sorted(types.items())))
print(f'  Hot Cues: {len(phrasing_result.cue_points)} mapped')

print('\n[CURATION ENGINE - Semantic Analysis]')
print(f'  Danceability: {curation_result.danceability:.2f}')
print(f'  Energy Type: {curation_result.energy_type}')
print(f'  Tags: {', '.join(curation_result.semantic_tags)}')

print('\n' + '='*80)
print('Feature extraction complete! All data ready for database storage.')
print('='*80)


COMPLETE FEATURE EXTRACTION FOR: Marrakech (Hermanez)

[GROOVE ENGINE - Rhythm Analysis]
  BPM: 129.20 (decimal precision)
  Beats: 1346 detected
  Swing: 1.00 (0=stiff, 1=groovy)
  Tempo Stable: True

[MOOD ENGINE - Harmonic Analysis]
  Musical Key: F#/Gb minor
  Camelot Key: 4A
  Brightness: 0.16 (0=dark, 1=bright)
  Key Confidence: 0.08

[PHRASING ENGINE - Structure Analysis]
  Total Segments: 94
  Segment Types: breakdown=89, build=1, drop=2, intro=1, outro=1
  Hot Cues: 91 mapped

[CURATION ENGINE - Semantic Analysis]
  Danceability: 0.60
  Energy Type: dynamic
  Tags: moderate-energy, peak-heavy, groovy, dark, techno, minimalist

Feature extraction complete! All data ready for database storage.
